In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure plots render inline inside Jupyter
%matplotlib inline 

import market_modelling.dsvi as dsvi
import market_modelling.svcj as svcj

from portfolio_models.linear_models import LongSPYWithTreasuryLadders
from portfolio_simulation import load_iv_surface

In [ ]:
spot, atm_iv, surface_chain = load_iv_surface("historical_data\\spx_iv_surface_2026_09_02.json")
closest_exp = min(surface_chain.keys())
strikes = surface_chain[closest_exp][0]
ivs = surface_chain[closest_exp][1]
svi = dsvi.DynamicSVI(strikes, ivs, spot, closest_exp)
print(f"Initial SPX: {spot:,.2f}")
print(f"Initial VIX: {atm_iv*100.0:.2f}%")
path_sim = svcj.SVCJSimulation(spot, atm_iv)
spx, vix, vix3m = path_sim.simulate_paths(17640, 1)
strategy = LongSPYWithTreasuryLadders(0.05, 0.95, 250_000)
return_path = strategy.run_simulation(
    spot_spx = spx[:, 0],
    spot_vix = vix[:, 0],
    vix3m = vix3m[:, 0],
    svi = svi,
    initial_nav = 8_400_000,
    days=17640,
    full_book=True)
book = strategy.transaction_book()

   

In [ ]:
spx_path = spx[:, 0] / 10.0
spx_df = pd.Series(spx_path)
ema30 = spx_df.ewm(span=20, adjust=False).mean().values
sma90 = spx_df.rolling(window=63).mean()
sma180 = spx_df.rolling(window=126).mean() 

# 1. Create mock data with wildly different scales (e.g., Stock Price vs. Trading Volume)
df = pd.DataFrame({
    "Day": range(0, len(spx_path)),
    "SPX": list(spx_path),
    "EMA30": ema30,
    "SMA90": sma90,
    "SMA180": sma180,
    "NAV": return_path
})
df.set_index("Day", inplace=True)

# 2. Set up the primary plot
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot the first time series (Left Y-axis)
color_price = "tab:blue"
ax1.set_xlabel("Day", fontsize=12)
ax1.set_ylabel("SPX Price ($)", color=color_price, fontsize=12)
line1 = ax1.plot(df.index, df["SPX"], color=color_price, linewidth=2, label="SPX Price ($)")
line2 = ax1.plot(df.index, df["EMA30"], color="tab:purple", linewidth=2, label="SPX EMA-30")
line3 = ax1.plot(df.index, df["SMA90"], color="tab:green", linewidth=2, label="SPX SMA-90")
line4 = ax1.plot(df.index, df["SMA180"], color="tab:orange", linewidth=2, label="SPX SMA-180")
ax1.tick_params(axis='y', labelcolor=color_price)
ax1.grid(True, linestyle="--", alpha=0.3)

# 3. Create a twin axis sharing the same X-axis
ax2 = ax1.twinx()  

# Plot the second time series (Right Y-axis)
color_volume = "tab:red"
ax2.set_ylabel("Portfolio NAV", color=color_volume, fontsize=12)
line4 = ax2.plot(df.index, df["NAV"], color=color_volume, linewidth=2, label="NAV")
ax2.tick_params(axis='y', labelcolor=color_volume)

# 4. Combine legends from both axes into a single box
lines = line1 + line2 + line3 + line4
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper left")

# Title and formatting
plt.title("Simulated SPX vs Portfolio", fontsize=14, fontweight="bold")
fig.autofmt_xdate() # Auto-rotate date labels
plt.tight_layout()

plt.show()

# 2. Set up the primary plot
fig, ax1 = plt.subplots(figsize=(12, 6))
vix_list = list(vix[:, 0])
vix3m_list = list(vix3m[:, 0])

df = pd.DataFrame({
    "Day": range(0, len(vix_list)),
    "VIX": vix_list,
    "VIX3M": vix3m_list
})

ax1.set_xlabel("Day", fontsize=12)
ax1.set_ylabel("Implied Volatility", fontsize=12)
line1 = ax1.plot(df.index, df["VIX"], color="tab:red", linewidth=2, label="VIX Index")
line2 = ax1.plot(df.index, df["VIX3M"], color="tab:orange", linewidth=2, label="VIX3M Index")
ax1.tick_params(axis='y', labelcolor=color_price)
ax1.grid(True, linestyle="--", alpha=0.3)

# 4. Combine legends from both axes into a single box
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper left")

# Title and formatting
plt.title("Volatility Simulation", fontsize=14, fontweight="bold")
fig.autofmt_xdate() # Auto-rotate date labels
plt.tight_layout()

plt.show()

# Massage the trade book to make it more presentable
sanitized_book = []
for i, row in enumerate(book):
    entry = book[i]
    symbol = ""
    position_size = ""
    rate = ""
    maturity = ""
    total = ""
    if "symbol" in entry:
        symbol = entry["symbol"]
    if "size" in entry:
        position_size = entry["size"]
    if "rate" in entry:
        rate = f"{entry["rate"]*100.0:.2f}%"
    if "maturity" in entry:
        maturity = f"{entry["maturity"]:.0f}"
    if "total" in entry:
        total = f"{entry["total"]:,.2f}"
        
    book_entry = {
        "Day": entry["day"] + 1,
        "NAV": f"{return_path[entry["day"]]:,.2f}",
        "SPY": f"{spx_path[entry["day"]]:,.2f}",
        "Trade": entry["trade"].title(),
        "Symbol": symbol,
        "Price": f"{entry["price"]:,.2f}",
        "Position Size": position_size,
        "Total": total,
        "Description": entry["description"],
        "Rate": rate,
        "Maturity": maturity
    }

    sanitized_book.append(book_entry)
    

trading_book = pd.DataFrame(sanitized_book)
# Tell pandas to show all rows
#pd.set_option('display.max_rows', None)
# Tell pandas to show all columns (just in case they are hidden too)
#pd.set_option('display.max_columns', None)

In [ ]:
trading_book